### Building a RAG System with LangChain and ChromaDB
#### Introduction
Retrieval-Augmented Generation (RAG) is a powerful technique that combines the capabilities of large language models with external knowledge retrieval. This notebook will walk you through building a complete RAG system using:

- LangChain: A framework for developing applications powered by language models
- ChromaDB: An open-source vector database for storing and retrieving embeddings
- OpenAI: For embeddings and language model (you can substitute with other providers)

In [13]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [14]:
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document

## vector store
## from langchain_community.vectorstores import Chroma
from langchain_chroma import Chroma
##utilities
import numpy as np
from typing import List, Tuple

In [15]:
# RAG Architecture Overview
print("""
RAG (Retrieval-Augmented Generation) Architecture:

1. Document Loading: Load documents from various sources
2. Document Splitting: Break documents into smaller chunks
3. Embedding Generation: Convert chunks into vector representations
4. Vector Storage: Store embeddings in ChromaDB
5. Query Processing: Convert user query to embedding
6. Similarity Search: Find relevant chunks from vector store
7. Context Augmentation: Combine retrieved chunks with query
8. Response Generation: LLM generates answer using context

Benefits of RAG:
- Reduces hallucinations
- Provides up-to-date information
- Allows citing sources
- Works with domain-specific knowledge
""")


RAG (Retrieval-Augmented Generation) Architecture:

1. Document Loading: Load documents from various sources
2. Document Splitting: Break documents into smaller chunks
3. Embedding Generation: Convert chunks into vector representations
4. Vector Storage: Store embeddings in ChromaDB
5. Query Processing: Convert user query to embedding
6. Similarity Search: Find relevant chunks from vector store
7. Context Augmentation: Combine retrieved chunks with query
8. Response Generation: LLM generates answer using context

Benefits of RAG:
- Reduces hallucinations
- Provides up-to-date information
- Allows citing sources
- Works with domain-specific knowledge



### 1. Sample Data

In [16]:
## create sample documents
sample_docs = [
    """
    Machine Learning Fundamentals
    
    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There are three main 
    types of machine learning: supervised learning, unsupervised learning, and reinforcement 
    learning. Supervised learning uses labeled data to train models, while unsupervised 
    learning finds patterns in unlabeled data. Reinforcement learning learns through 
    interaction with an environment using rewards and penalties.
    """,
    
    """
    Deep Learning and Neural Networks
    
    Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks are inspired by the human brain and consist of layers of interconnected 
    nodes. Deep learning has revolutionized fields like computer vision, natural language 
    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly 
    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers 
    excel at sequential data processing.
    """,
    
    """
    Natural Language Processing (NLP)
    
    NLP is a field of AI that focuses on the interaction between computers and human language. 
    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, 
    machine translation, and question answering. Modern NLP heavily relies on transformer 
    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand 
    context and relationships between words in text.
    """
]

sample_docs


['\n    Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through \n    interaction with an environment using rewards and penalties.\n    ',
 '\n    Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly \n    effective f

In [17]:
##save the documents to text files
import tempfile
temp_dir=tempfile.mkdtemp()

for i, doc in enumerate(sample_docs):
    with open(f"{temp_dir}/doc_{i}.txt","w") as f:
        f.write(doc)
print(f"Documents saved to: {temp_dir}")

Documents saved to: /tmp/tmp4y8a9yfw


### 2. Document Loading

In [18]:
from langchain_community.document_loaders import DirectoryLoader
#Load documents from the temporary directory
loader = DirectoryLoader(
    temp_dir,
    glob="*.txt",
    loader_cls= TextLoader,
    loader_kwargs={"encoding":"utf-8"}
)

In [19]:
documents =loader.load()
print(f"Loaded {len(documents)} documents.")
print("Sample Document:")
print(documents[0].page_content[:500])

Loaded 3 documents.
Sample Document:

    Natural Language Processing (NLP)

    NLP is a field of AI that focuses on the interaction between computers and human language. 
    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, 
    machine translation, and question answering. Modern NLP heavily relies on transformer 
    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand 
    context and relationships between words in text.
    


### Document Splitting

In [20]:
# Intialize text splitter
text_splitter= RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function= len,
    separators=[" "]
)
chunks= text_splitter.split_documents(documents)
print(f"Total Chunks Created: {len(chunks)} from {len(documents)} documents.")
print("Sample Chunk:")
print(chunks[0].page_content[:500])
print(f"Chunk 1 Metadata: {chunks[0].metadata}")
print(chunks[1].page_content[:500])
print(f"Chunk 2 Metadata: {chunks[1].metadata}")


Total Chunks Created: 5 from 3 documents.
Sample Chunk:
Natural Language Processing (NLP)

    NLP is a field of AI that focuses on the interaction between computers and human language. 
    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, 
    machine translation, and question answering. Modern NLP heavily relies on transformer 
    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand 
    context and relationships between words in text.
Chunk 1 Metadata: {'source': '/tmp/tmp4y8a9yfw/doc_2.txt'}
Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There are three main 
    types of machine learning: supervised learning, unsupervised learning, and reinforcement 
    learning. Supervised learning uses labeled data to train models, while unsupervised 
    learning finds pat

### Embedding Models

In [21]:
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [22]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
embeddings

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x7f59b6b8ed50>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x7f59b6b8f610>, model='text-embedding-3-small', dimensions=None, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base='https://models.inference.ai.azure.com', openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

In [23]:
sample_query="Machine learning"
vector = embeddings.embed_query(sample_query)
vector

[-0.0093231201171875,
 -0.002307891845703125,
 -0.0008492469787597656,
 -0.02191162109375,
 0.042236328125,
 0.01093292236328125,
 0.0076904296875,
 0.0396728515625,
 -0.0195770263671875,
 0.0225067138671875,
 -0.005756378173828125,
 -0.050384521484375,
 -0.03179931640625,
 -0.020172119140625,
 0.013153076171875,
 0.01849365234375,
 -0.0081634521484375,
 -0.035125732421875,
 0.02081298828125,
 0.006153106689453125,
 0.0164794921875,
 0.0197906494140625,
 0.036865234375,
 -0.02386474609375,
 0.037994384765625,
 -0.01519775390625,
 0.03448486328125,
 0.031280517578125,
 0.00821685791015625,
 -0.0287017822265625,
 -0.0283966064453125,
 -0.018890380859375,
 -0.042236328125,
 -0.0048675537109375,
 0.031707763671875,
 -0.0132598876953125,
 -0.00872802734375,
 0.0355224609375,
 -0.04864501953125,
 0.0168609619140625,
 -0.024627685546875,
 -0.027008056640625,
 0.0163726806640625,
 0.07183837890625,
 -0.0156707763671875,
 -0.0250091552734375,
 -0.042877197265625,
 -0.0439453125,
 0.009445190429

### INitialize the ChromaDB Store and Store the chunks in Vector Representation

In [24]:
## Create a chromadb directory
# make a directory for chromadb if it doesn't exist
if not os.path.exists("./chromadb"):
    os.makedirs("./chromadb")

persist_dir ="./chromadb"
vectorstore = Chroma(
    persist_directory=persist_dir,
    embedding_function=embeddings,
    collection_name="chroma_documents"
)
##vectorstore.add_documents(chunks)
print(f"Created {vectorstore._collection.count()} vectors")
print(f"Stored at {persist_dir}")

Created 5 vectors
Stored at ./chromadb


### Similarity Search

In [25]:
sample_query="What is NLP"
similar_docs = vectorstore.similarity_search(sample_query,k=3)
similar_docs

[Document(id='5239c6a1-c15e-4c7c-a25e-f3ce7b4987f1', metadata={'source': '/tmp/tmp4y8a9yfw/doc_2.txt'}, page_content='Natural Language Processing (NLP)\n\n    NLP is a field of AI that focuses on the interaction between computers and human language. \n    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, \n    machine translation, and question answering. Modern NLP heavily relies on transformer \n    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand \n    context and relationships between words in text.'),
 Document(id='ddd58660-2b22-4f28-b765-ed85bbe95dcf', metadata={'source': '/tmp/tmp4y8a9yfw/doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer visi

In [26]:
sample_query="What is deep learning"
similar_docs = vectorstore.similarity_search(sample_query,k=3)
similar_docs

[Document(id='ddd58660-2b22-4f28-b765-ed85bbe95dcf', metadata={'source': '/tmp/tmp4y8a9yfw/doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly \n    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers'),
 Document(id='44214dfd-6320-45a8-80a8-9c0983840bc6', metadata={'source': '/tmp/tmp4y8a9yfw/doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised lea

In [27]:
print(f"Sample Query: {sample_query}")
print(f"Top {len(similar_docs)} Similar Chunks:")
for i, doc in enumerate(similar_docs):
    print(f"\nChunk {i+1} Metadata: {doc.metadata}")
    print(f"Chunk {i+1} Content: {doc.page_content[:500]}...")

Sample Query: What is deep learning
Top 3 Similar Chunks:

Chunk 1 Metadata: {'source': '/tmp/tmp4y8a9yfw/doc_1.txt'}
Chunk 1 Content: Deep Learning and Neural Networks

    Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks are inspired by the human brain and consist of layers of interconnected 
    nodes. Deep learning has revolutionized fields like computer vision, natural language 
    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly 
    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers...

Chunk 2 Metadata: {'source': '/tmp/tmp4y8a9yfw/doc_0.txt'}
Chunk 2 Content: Machine Learning Fundamentals

    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There are three main 
    types of machine learning: supervised learning, unsupervised learning, and 

In [28]:
results_scores = vectorstore.similarity_search_with_score(sample_query,k=3)
results_scores

[(Document(id='ddd58660-2b22-4f28-b765-ed85bbe95dcf', metadata={'source': '/tmp/tmp4y8a9yfw/doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly \n    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers'),
  0.7544082403182983),
 (Document(id='44214dfd-6320-45a8-80a8-9c0983840bc6', metadata={'source': '/tmp/tmp4y8a9yfw/doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine

#### Understanding Similarity Scores
The similarity score represents how closely related a document chunk is to your query. The scoring depends on the distance metric used:

ChromaDB default: Uses L2 distance (Euclidean distance)

- Lower scores = MORE similar (closer in vector space)
- Score of 0 = identical vectors
- Typical range: 0 to 2 (but can be higher)


Cosine similarity (if configured):

- Higher scores = MORE similar
- Range: -1 to 1 (1 being identical)

### Initialize LLM, RAG Chain, Prompt Template, Query the RAG System

In [29]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
import os
load_dotenv()
llm = ChatOpenAI(
    model_name="gpt-4.1",
    temperature=0
)

In [30]:
test_response= llm.invoke("What is deep learning")
print(test_response)

content='**Deep learning** is a subset of **machine learning** and **artificial intelligence (AI)** that uses algorithms inspired by the structure and function of the brain, called **artificial neural networks**. Deep learning models automatically learn to extract features and patterns from large amounts of data, enabling them to perform complex tasks such as image recognition, speech processing, natural language understanding, and more.\n\n### Key Points:\n- **Neural Networks:** Deep learning uses multi-layered neural networks (often called "deep" because they have many layers) to process data.\n- **Learning from Data:** These models learn directly from raw data, often requiring large datasets and significant computational power.\n- **Applications:** Deep learning powers technologies like self-driving cars, virtual assistants, facial recognition, translation services, and medical diagnostics.\n\n### Example:\n- **Image Recognition:** A deep learning model can learn to identify objects

In [31]:
from langchain.chat_models.base import init_chat_model
llm = init_chat_model(
    model="gpt-4o",
    model_provider="openai"
)
test_response= llm.invoke("What is deep learning")
print(test_response)

content='Deep learning is a subset of **machine learning (ML)**, which itself is a branch of **artificial intelligence (AI)**. It focuses on algorithms inspired by the structure and functioning of the human brain, known as **artificial neural networks**. These networks are designed to process, analyze, and make predictions based on large and complex datasets.\n\n### Key Characteristics of Deep Learning:\n1. **Neural Networks:** Deep learning models are composed of multiple layers of artificial neural networks. These networks process data iteratively, with each layer extracting progressively more abstract features. The idea is loosely based on how the human brain processes information.\n\n2. **Representation Learning:** Unlike traditional ML algorithms, which often rely on manual feature extraction (human involvement in deciding what features to use), deep learning automatically learns high-level features from raw data. This makes it highly effective for tasks involving unstructured dat

### Modern RAG Chain

In [32]:
from langchain_classic.chains import create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate
# import combine documents
from langchain_classic.chains.combine_documents import create_stuff_documents_chain


In [33]:
# convert vectore store to retriever
retrieve = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)
retrieve

VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x7f59b6b8f9d0>, search_kwargs={'k': 3})

In [34]:
#create prompt template
system_prompt = """
You are an assistant fir quaestion answering tasks.
Use the following pieces of retrieval context to anwser the question.
If you don't know the answer, just say that you don't know, don't try to make up an answer.

Context:{context}
"""

prompt = ChatPromptTemplate.from_messages([
    {"role":"system", "content": system_prompt},
    {"role":"human", "content":"{input}"}
])

In [35]:
prompt

ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="\nYou are an assistant fir quaestion answering tasks.\nUse the following pieces of retrieval context to anwser the question.\nIf you don't know the answer, just say that you don't know, don't try to make up an answer.\n\nContext:{context}\n"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])

##### What is create_stuff_documents_chain?
create_stuff_documents_chain creates a chain that "stuffs" (inserts) all retrieved documents into a single prompt and sends it to the LLM. It's called "stuff" because it literally stuffs all the documents into the context window at once.


#### What is create_retrieval_chain?
create_retrieval_chain is a function that combines a retriever (which fetches relevant documents) with a document chain (which processes those documents with an LLM) to create a complete RAG pipeline.

In [36]:
#create a document chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
document_chain = create_stuff_documents_chain(llm=llm, prompt=prompt)
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="\nYou are an assistant fir quaestion answering tasks.\nUse the following pieces of retrieval context to anwser the question.\nIf you don't know the answer, just say that you don't know, don't try to make up an answer.\n\nContext:{context}\n"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])
| ChatOpenAI(output_version=None, profile={'name': 'GPT-4o', 'release_date': '2024-05-13', 'last_updated': '2024-08-06', 'open_weights': False, 'max_input_toke

This chain:

- This creates a chain that takes retrieved documents.
- It inserts their text into your prompt’s {context} placeholder.
- Then it sends the full prompt (context + user question) to llm.
- Output is the model’s answer text.

In [37]:
### Create final Rag chain
rag_chain = create_retrieval_chain(retrieve, document_chain)
rag_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x7f59b6b8f9d0>, search_kwargs={'k': 3}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="\nYou are an assistant fir quaestion answering tasks.\nUse the following pieces of retrieval context to anwser the question.\nIf you don't know 

This wraps two steps together:
- Step A: retrieve finds relevant chunks from your vector store.
- Step B: document_chain uses those chunks to generate an answer.
- So now you have one callable chain that does retrieval + generation end-to-end.

1. First chain (create_stuff_documents_chain)
 - It combines LLM + prompt template.
 - It expects context and input.
 - It does not retrieve documents itself.
 - It “maps” retrieved docs into {context} and asks the LLM to answer.

2. Second chain (create_retrieval_chain)
 - It combines retriever + first chain.
 - At run time (invoke), it retrieves relevant docs from the vector store.
 - Then it passes those docs as context into the first chain.
So you get one end-to-end RAG pipeline: question -> retrieve context -> generate answer.
So your core idea is right: first chain is answer-generation with context slot, second chain adds retrieval so that slot gets real relevant documents automatically.

In [38]:
response = rag_chain.invoke({"input": "What is deep learning?"})
response


{'input': 'What is deep learning?',
 'context': [Document(id='ddd58660-2b22-4f28-b765-ed85bbe95dcf', metadata={'source': '/tmp/tmp4y8a9yfw/doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly \n    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers'),
  Document(id='44214dfd-6320-45a8-80a8-9c0983840bc6', metadata={'source': '/tmp/tmp4y8a9yfw/doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three mai

In [39]:
def query_rag(query:str):
    print(f"Query: {query}")
    print("-"*50)

    response = rag_chain.invoke({"input": query})
    print(f"Response: {response['answer']}")

    for i, doc in enumerate(response["context"]):
        print(f"\n Source {i+1} Metadata: {doc.metadata}")
        print(f" Source {i+1} Content: {doc.page_content[:500]}...")

    return response

In [40]:
query_rag("What machine learning is?")

Query: What machine learning is?
--------------------------------------------------
Response: Machine learning is a subset of artificial intelligence (AI) that enables systems to learn and improve from experience without being explicitly programmed. It focuses on developing algorithms and models that can analyze data, identify patterns, and make decisions or predictions based on that data. 

There are three main types of machine learning:

1. **Supervised Learning**: In this approach, models are trained using labeled data, where the input data comes with corresponding output labels (e.g., predicting house prices based on features like size and location).

2. **Unsupervised Learning**: In this approach, models work on unlabeled data to identify patterns, structures, or groupings within the data (e.g., clustering customers based on purchasing behavior).

3. **Reinforcement Learning**: This approach involves learning through interaction with an environment, where an agent takes actions to

{'input': 'What machine learning is?',
 'context': [Document(id='44214dfd-6320-45a8-80a8-9c0983840bc6', metadata={'source': '/tmp/tmp4y8a9yfw/doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through'),
  Document(id='ddd58660-2b22-4f28-b765-ed85bbe95dcf', metadata={'source': '/tmp/tmp4y8a9yfw/doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n   

### Create RAG Chain Alternative - Using LCEL(LangChain Expression Language)

In [41]:
#creating RAG chain without using utility functions
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableSerializable, RunnableParallel

In [42]:
#Creating a a custom prompt
custom_prompt = ChatPromptTemplate.from_template("""
Use the following to context to answer the question. 
If you don't know the answer, just say that you don't know, don't try to make up an answer.
Provide specific details from the context to support your answer.

Context: {context}
Question: {question}
Answer:
"""
)
custom_prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="\nUse the following to context to answer the question. \nIf you don't know the answer, just say that you don't know, don't try to make up an answer.\nProvide specific details from the context to support your answer.\n\nContext: {context}\nQuestion: {question}\nAnswer:\n"), additional_kwargs={})])

In [43]:
#format the output document for the custom prompt
def format_docs(docs):
    return "\n\n".join([f"Source: {doc.metadata['source']}\nContent: {doc.page_content}" for doc in docs])


In [ ]:
#Build chain using LCEL
rag_chain_lcel = (
    {
        "context": retrieve | format_docs,
        "question": RunnablePassthrough()
    }
    | custom_prompt 
    | llm 
    | StrOutputParser()
)
rag_chain_lcel

{
  context: VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x7f59b6b8f9d0>, search_kwargs={'k': 3})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="\nUse the following to context to answer the question. \nIf you don't know the answer, just say that you don't know, don't try to make up an answer.\nProvide specific details from the context to support your answer.\n\nContext: {context}\nQuestion: {question}\nAnswer:\n"), additional_kwargs={})])
| ChatOpenAI(output_version=None, profile={'name': 'GPT-4o', 'release_date': '2024-05-13', 'last_updated': '2024-08-06', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'tex

In [45]:
response = rag_chain_lcel.invoke("What is deep learning?")
print(f"Response: {response}")

Response: Deep learning is a subset of machine learning based on artificial neural networks. These networks are inspired by the human brain and consist of layers of interconnected nodes. Deep learning has revolutionized fields like computer vision, natural language processing, and speech recognition.


In [46]:
format_docs(retrieve.invoke("What is deep learning?"))

'Source: /tmp/tmp4y8a9yfw/doc_1.txt\nContent: Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly \n    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers\n\nSource: /tmp/tmp4y8a9yfw/doc_0.txt\nContent: Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    l

### Add documents to the existing vector store

In [47]:
vectorstore

In [48]:
# vector store with new documents
new_document = """
Generative Artificial Intelligence (AI)

Generative AI refers to a class of artificial intelligence models that can generate new content, such as text, 
images, or audio, based on patterns learned from existing data. These models use techniques 
like deep learning and transformers to create outputs that are often indistinguishable from human-generated content. Generative AI has applications in various fields, including creative writing, art generation, and even code synthesis.

Generative AI models, such as GPT-3 and DALL-E, have demonstrated remarkable capabilities in generating 
coherent text and realistic images. These models are trained on vast amounts of data and can produce 
creative outputs that mimic human style and creativity. However, they also raise ethical concerns 
regarding misinformation, bias, and the potential for misuse.

Generative AI is a rapidly evolving field that has expanded beyond traditional text generation to include a 
wide range of applications. With the advent of powerful models and techniques, generative AI can now create not only text but also images, 
videos, music, code, and even complex documents. This has opened up new possibilities for creativity and 
productivity across various domains. Generative AI can be used to schedule events, connect with other tools, 
and even create agentic AI that can perform tasks autonomously. As the technology continues to advance, 
we can expect to see even more innovative applications of generative AI in the future.


As the field of generative AI continues to evolve, it is crucial to balance innovation with ethical 
considerations. Responsible development and deployment of generative AI can lead to incredible 
advancements while minimizing potential risks. By fostering collaboration between researchers, 
policymakers, and industry leaders, we can ensure that generative AI is used for the betterment 
of society while addressing concerns related to misinformation, bias, and misuse. The future of 
generative AI holds immense promise, and with careful stewardship, it can be a powerful tool 
for creativity, productivity, and positive impact across various domains.
"""

In [49]:
new_document


'\nGenerative Artificial Intelligence (AI)\n\nGenerative AI refers to a class of artificial intelligence models that can generate new content, such as text, \nimages, or audio, based on patterns learned from existing data. These models use techniques \nlike deep learning and transformers to create outputs that are often indistinguishable from human-generated content. Generative AI has applications in various fields, including creative writing, art generation, and even code synthesis.\n\nGenerative AI models, such as GPT-3 and DALL-E, have demonstrated remarkable capabilities in generating \ncoherent text and realistic images. These models are trained on vast amounts of data and can produce \ncreative outputs that mimic human style and creativity. However, they also raise ethical concerns \nregarding misinformation, bias, and the potential for misuse.\n\nGenerative AI is a rapidly evolving field that has expanded beyond traditional text generation to include a \nwide range of applicatio

In [50]:
new_doc = Document(
    page_content=new_document,
    metadata={
        "source": "new_doc.txt",
        "topic": "Generative AI",
        "author": "John Doe"
    }
)
new_doc

Document(metadata={'source': 'new_doc.txt', 'topic': 'Generative AI', 'author': 'John Doe'}, page_content='\nGenerative Artificial Intelligence (AI)\n\nGenerative AI refers to a class of artificial intelligence models that can generate new content, such as text, \nimages, or audio, based on patterns learned from existing data. These models use techniques \nlike deep learning and transformers to create outputs that are often indistinguishable from human-generated content. Generative AI has applications in various fields, including creative writing, art generation, and even code synthesis.\n\nGenerative AI models, such as GPT-3 and DALL-E, have demonstrated remarkable capabilities in generating \ncoherent text and realistic images. These models are trained on vast amounts of data and can produce \ncreative outputs that mimic human style and creativity. However, they also raise ethical concerns \nregarding misinformation, bias, and the potential for misuse.\n\nGenerative AI is a rapidly e

### Add new document to vector store

#### split_document vs split_text

goal: Explain split_documents vs split_text for text splitters, in Markdown with examples.
What They Do

 - split_documents: Accepts a list of Document objects and returns a list of chunked Document objects (keeps metadata).
 - split_text: Accepts a single raw str and returns a list of text chunks (strings), no document metadata.

#### Key Differences

 - Input type: split_documents → list of Document; split_text → str.
 - Output type: split_documents → list of Document (with page_content + metadata); split_text → list of str.
 - Metadata: split_documents preserves and can attach/propagate metadata; split_text loses metadata.
 - Use-case: split_documents for pipelines where source attribution matters (citations, source fields).     split_text for quick ad-hoc splitting of plain strings.

In [51]:
## Splitting the document into chunks
new_chunks = text_splitter.split_documents([new_doc])
new_chunks

[Document(metadata={'source': 'new_doc.txt', 'topic': 'Generative AI', 'author': 'John Doe'}, page_content='Generative Artificial Intelligence (AI)\n\nGenerative AI refers to a class of artificial intelligence models that can generate new content, such as text, \nimages, or audio, based on patterns learned from existing data. These models use techniques \nlike deep learning and transformers to create outputs that are often indistinguishable from human-generated content. Generative AI has applications in various fields, including creative writing, art generation, and even code synthesis.\n\nGenerative AI'),
 Document(metadata={'source': 'new_doc.txt', 'topic': 'Generative AI', 'author': 'John Doe'}, page_content='and even code synthesis.\n\nGenerative AI models, such as GPT-3 and DALL-E, have demonstrated remarkable capabilities in generating \ncoherent text and realistic images. These models are trained on vast amounts of data and can produce \ncreative outputs that mimic human style a

In [53]:
## Add new document to vector store
vectorstore.add_documents(new_chunks)

['525def76-06e3-4ea1-825b-73c228a7d136',
 '240ddcc7-dfeb-473f-9ea4-e1da14e8ca94',
 '13ff555c-ff19-4390-b048-3decf415d2a7',
 '21171319-26c2-4a57-a954-9e8fee996122',
 '958579e2-39a0-42b4-9ca6-12a725b1566b']

In [61]:
print(f"New Chunks {len(new_chunks)}. Total Vectors after adding new document: {vectorstore._collection.count()}")

New Chunks 5. Total Vectors after adding new document: 10


In [62]:
## query with the updated vector store
new_question = "What kind of things we can do with generative AI?"
result = query_rag(new_question)
result

Query: What kind of things we can do with generative AI?
--------------------------------------------------
Response: Generative AI has evolved to offer a wide range of applications across various domains, enabling creativity, productivity, and innovation. Here are some examples of what you can do with generative AI:

1. **Text Generation:**
   - Generate human-like written content, such as essays, articles, poetry, and stories.
   - Draft professional emails, resumes, or creative writing projects.
   - Chatbots and virtual assistants for customer support or digital interaction.

2. **Image Generation:**
   - Create realistic or stylized images from textual descriptions (e.g., DALL-E).
   - Design art, illustrations, logos, or other visual content.
   - Enhance, modify, or restore existing photos.

3. **Video Creation:**
   - Generate or edit videos, animations, or deepfakes.
   - Create video ads, promotional content, and cinematic sequences.

4. **Music and Audio:**
   - Compose new 

{'input': 'What kind of things we can do with generative AI?',
 'context': [Document(id='13ff555c-ff19-4390-b048-3decf415d2a7', metadata={'author': 'John Doe', 'source': 'new_doc.txt', 'topic': 'Generative AI'}, page_content='expanded beyond traditional text generation to include a \nwide range of applications. With the advent of powerful models and techniques, generative AI can now create not only text but also images, \nvideos, music, code, and even complex documents. This has opened up new possibilities for creativity and \nproductivity across various domains. Generative AI can be used to schedule events, connect with other tools, \nand even create agentic AI that can perform tasks autonomously. As the technology'),
  Document(id='525def76-06e3-4ea1-825b-73c228a7d136', metadata={'source': 'new_doc.txt', 'author': 'John Doe', 'topic': 'Generative AI'}, page_content='Generative Artificial Intelligence (AI)\n\nGenerative AI refers to a class of artificial intelligence models that can g

### Advanced RAG Concepts
 - Advanced rag, conversational memory,the previous rag that we built dont have any memory of the previous interactions. 
 - We can add a conversational memory to the RAG chain to make it more interactive and context-aware. 
 - This way, the model can remember previous questions and answers, allowing for a more natural conversation flow.

 Example:
 - Q1 - We asked what is gnerative ai, 
 - Q2 - We asked what kind of things we can do with generative ai,
 - so the system should be able to remember the previous conversation and provide a more detailed answer based on the new document we added to the vector store.

#### Conversational Memory
- create_history_aware_retriever: Makes the retriever understand conversation context
- MessagesPlaceholder: Placeholder for chat history in prompts
- HHumanMessage/AIMessage: Structure message types for conversation history

In [81]:
from langchain_classic.chains import create_history_aware_retriever
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

In [82]:
## create a prompt with conversation history
contextual_system_prompt ="""
Given a chat history and the latest question which might refer to previous conversation, 
formulate a standalone question which can be answered without the need of chat history.
DO NOT answer the question, just reformulate it if needed or return the question as is if it doesn't refer to previous conversation.
"""

contextual_prompt = ChatPromptTemplate.from_messages([
    {"role":"system", "content": contextual_system_prompt},
    MessagesPlaceholder(variable_name="chat_history"),
    {"role":"user", "content": "{input}"}
])

In [84]:
## create history aware retriever
history_aware_retriever = create_history_aware_retriever(
    llm, retrieve, contextual_prompt
)
history_aware_retriever

RunnableBinding(bound=RunnableBranch(branches=[(RunnableLambda(lambda x: not x.get('chat_history', False)), RunnableLambda(lambda x: x['input'])
| VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x7f59b6b8f9d0>, search_kwargs={'k': 3}))], default=ChatPromptTemplate(input_variables=['chat_history', 'input'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageCh

In [83]:
# Create a new document chain with history
qa_system_prompt ="""
You are an assistant for question-answering tasks. 
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know, don't try to make up an answer.
Use three sentences maximum to answer the question and be concise.

Context: {context}
"""

qa_prompt = ChatPromptTemplate.from_messages([
    {"role":"system", "content": qa_system_prompt},
    {"role":"user", "content": "{input}"}
])

In [78]:
# Create a new document chain with the custom QA prompt
question_answer_chain = create_stuff_documents_chain(llm,qa_prompt)
question_answer_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="\nYou are an assistant for question-answering tasks. \nUse the following pieces of retrieved context to answer the question.\nIf you don't know the answer, just say that you don't know, don't try to make up an answer.\nUse three sentences maximum to answer the question and be concise.\n\nContext: {context}\n"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])
| ChatOpenAI(output_version=None, profile={'name': 'GPT-4o', 'release_date': '2024-05-13',

In [87]:
# Create a conversation RAG chain
conversation_rag_chain = create_retrieval_chain(
    history_aware_retriever,
    question_answer_chain
)
conversation_rag_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableBranch(branches=[(RunnableLambda(lambda x: not x.get('chat_history', False)), RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x7f59b6b8f9d0>, search_kwargs={'k': 3}))], default=ChatPromptTemplate(input_variables=['chat_history', 'input'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annot

In [88]:
chat_history = []
result1 = conversation_rag_chain.invoke({
    "input": "What is generative AI?", 
    "chat_history": chat_history
})
chat_history.extend(
    [
        HumanMessage(content="What is generative AI?"),
        AIMessage(content=result1["answer"])
    ]
)
print(f"Question: What is generative AI?")
print(f"Answer: {result1['answer']}")

Question: What is generative AI?
Answer: Generative AI refers to a class of artificial intelligence models that create new content, such as text, images, music, or code, by learning patterns from existing data. It uses techniques like deep learning and models like GPT-3 and DALL-E to produce outputs often indistinguishable from human-created content. This technology has broad applications, including creative writing, art, and autonomous task execution, but it also raises ethical concerns.


In [90]:
## Follow up question
result2 = conversation_rag_chain.invoke({
    "input": "What are the applications of generative AI?", 
    "chat_history": chat_history
})
print(f"\nQuestion: What are its applications?")
print(f"Answer: {result2['answer']}")


Question: What are its applications?
Answer: Generative AI has a wide range of applications, including creating text, images, videos, music, code, and complex documents. It is used in fields like creative writing, art generation, software development, and content creation. Additionally, generative AI can perform tasks such as event scheduling, connecting with other tools, and executing tasks autonomously.
